In [ ]:
import os  
import asyncio
from io import BytesIO

import pandas as pd
import gspread
from google.oauth2.service_account import Credentials

from datetime import datetime as dt
from datetime import timedelta

from aiogram import Bot, Dispatcher, F, Router
from aiogram.filters import CommandStart, Command
from aiogram.types import Message, CallbackQuery, InlineKeyboardMarkup, InlineKeyboardButton
from aiogram.utils.keyboard import InlineKeyboardBuilder

import matplotlib.pyplot as plt
import pandas as pd

In [2]:
scopes = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
]

creds = Credentials.from_service_account_file('credentials.json', scopes=scopes)
client = gspread.authorize(creds)

yesterday = dt.today() - timedelta(days=1)
months_dict = {
    1: "Январь",
    2: "Февраль",
    3: "Март",
    4: "Апрель",
    5: "Май",
    6: "Июнь",
    7: "Июль",
    8: "Август",
    9: "Сентябрь",
    10: "Октябрь",
    11: "Ноябрь",
    12: "Декабрь"
}
year = yesterday.year
month = months_dict.get(yesterday.month)
day = yesterday.day

full_date  = yesterday.strftime('%d.%m.%Y')
full_date

'14.09.2026'

In [3]:
key = next((k for k, v in months_dict.items() if v == month), None)
past_month = months_dict.get(key - 1)
past_month

'Август'

In [4]:
import re

In [5]:
def get_sample_data():
    sheet = client.open_by_key('1o1mIcsXQht1NFhgsq7CMKI3derC8xOSrRgGC9GYu144').worksheet(f'{month} {year}')
    data = sheet.get_all_values()
    df = pd.DataFrame(data)
    df = df.loc[3:]
    df_basic = df.copy()
    df_basic.columns = df.iloc[0]
    df_basic = df_basic[1:].reset_index(drop=True)
    df_basic = df_basic.set_index('Дата')

    

    return df_basic

In [6]:
def get_ya_metrik(all_data):
    
    return all_data().iloc[:, :10][['номер недели', 'Трафик', 'Уникальные', 'vs LY', 'Переход в каталог', 'Положил в корзину', 'Оформил заказ']]

In [7]:
def get_basic_data(all_data):
    return all_data()[['План Руб', 'План Заказы', 'Заказы Сайт, шт', 'Сумма заказов РУБ']]

In [8]:
def get_reg_info(all_data):
    return all_data().iloc[:, 18:21][[ 'ШТ', 'vs LY']]

In [9]:
def get_cerf_info(all_data):    
    return all_data()[['Сертификаты, шт', 'Сертификаты, Руб']]

In [10]:
type(get_sample_data()['Трафик'].loc['13.09.2026'])

str

In [11]:
df = get_ya_metrik(get_sample_data)
day_ya_info = df.loc[full_date]

day_ya_info = (day_ya_info.astype('str')  
                        .str.replace(r'\s+', '', regex=True)
                        .astype('float32')) # Убираем лишние пробелы во всей таблице и переводим строчные даннын в числовые

In [12]:
day_ya_info


3
номер недели           38.0
Трафик                990.0
Уникальные            774.0
vs LY                1900.0
Переход в каталог     315.0
Положил в корзину      94.0
Оформил заказ          30.0
Name: 14.09.2026, dtype: float32

In [13]:
traffic_vs_LY = 100 * day_ya_info['Уникальные']/day_ya_info['vs LY']
CTR_to_catalog = 100 * day_ya_info['Переход в каталог']/day_ya_info['Уникальные']
CTR_to_basket = 100 * day_ya_info['Положил в корзину']/day_ya_info['Уникальные']
CTR_to_order = 100 * day_ya_info['Оформил заказ']/day_ya_info['Уникальные']
print(f' Доля траффика от прошлогоднего: {traffic_vs_LY:,.2f}% \n \
rerere')


 Доля траффика от прошлогоднего: 40.74% 
 rerere


In [14]:
print(f' Доля траффика от прошлогоднего: {traffic_vs_LY:,.2f}% \n \
    CTR перехода в каталог: {CTR_to_catalog:,.2f}% \n \
    CTR добавления в корзину: {CTR_to_basket:,.2f}% \n \
    CTR создания заказа: {CTR_to_order:,.2f}%')

 Доля траффика от прошлогоднего: 40.74% 
     CTR перехода в каталог: 40.70% 
     CTR добавления в корзину: 12.14% 
     CTR создания заказа: 3.88%


In [ ]:
df = get_basic_data(get_sample_data)

day_sales_info = df.loc[full_date]
type(day_sales_info['Сумма заказов РУБ'])

str

str

In [30]:
full_date_previous = dt.strptime(full_date, '%d.%m.%Y').date().strftime('%d.%m.%Y')
while df.loc[full_date_previous]['Сумма заказов РУБ'] == '':
    full_date_previous = full_date_previous - timedelta(days=1)

full_date_previous   

TypeError: unsupported operand type(s) for -: 'str' and 'datetime.timedelta'

In [ ]:

day_sales_info = (day_sales_info.astype('str')  
                        .str.replace(r'\s+', '', regex=True)
                        .astype('float32')) # Убираем лишние пробелы во всей таблице и переводим строчные даннын в числовые

type(day_sales_info['Сумма заказов РУБ'])

ValueError: could not convert string to float: ''